In [ ]:
!pip install transformers accelerate bitsandbytes scikit-learn sentence-transformers numpy scipy requests

import torch
import requests
from io import StringIO
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist, squareform
from scipy.linalg import eigh
import re
import pandas as pd
from IPython.display import display

In [ ]:
# Загрузка Qwen

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Модель для эмбеддингов
embedder = SentenceTransformer('intfloat/multilingual-e5-small') # slava
#embedder = SentenceTransformer('all-MiniLM-L6-v2') #tqa


In [ ]:
# Класс детектора
class HallucinationDetector:
    def __init__(self, model, tokenizer, embedder):
        self.model = model
        self.tokenizer = tokenizer
        self.embedder = embedder

        # Пороги для метрик (По умолчанию)
        self.thresholds = {
            'semantic_entropy': 0.4,
            'perplexity': 50,
            'token_entropy': 4.0,
            'self_confidence': 0.5,
            'eigen_score': 0.3,
            'sar_score': 0.6,
            'num_sets': 2,
            'verbalized_uncertainty': 0.5,
            'embedding_similarity': 0.7
        }

    def generate_responses(self, question, context=None, num_responses=5, temperature=0.7, max_new_tokens=256):
        if context:
            prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer:"
        else:
            prompt = f"Question: {question}\nAnswer:"

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        responses = []
        for _ in range(num_responses):
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=True,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            answer = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            responses.append(answer.strip())
        return responses


    def semantic_entropy(self, responses):
        if len(responses) < 2:
            return 0.0
        emb = self.embedder.encode(responses)
        sim_matrix = cosine_similarity(emb)
        np.fill_diagonal(sim_matrix, 0)
        mean_sim = sim_matrix.sum() / (len(responses) * (len(responses) - 1))
        return 1.0 - mean_sim

    def perplexity(self, question, answer, context=None):
        if context:
            prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer:"
        else:
            prompt = f"Question: {question}\nAnswer:"
        full_text = prompt + answer
        inputs = self.tokenizer(full_text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss.item()
        return np.exp(loss)

    def token_entropy(self, question, context=None, max_new_tokens=256):
        if context:
            prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer:"
        else:
            prompt = f"Question: {question}\nAnswer:"

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                return_dict_in_generate=True,
                output_scores=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        generated_ids = outputs.sequences[0][inputs.input_ids.shape[1]:]
        answer = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        entropies = []
        for scores in outputs.scores:
            probs = torch.softmax(scores[0], dim=-1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-10)).item()
            entropies.append(entropy)
        avg_entropy = np.mean(entropies) if entropies else 0.0
        return answer, avg_entropy

    def self_check(self, question, answer):
        prompt = f"""Question: {question}
Proposed answer: {answer}

On a scale from 0 to 1, how confident are you that this answer is correct?
0 = not confident at all, 1 = completely confident.
Answer with only a number (e.g., 0.85)."""
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.0,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        match = re.search(r'0\.\d+|1\.0|0|1', response)
        if match:
            confidence = float(match.group())
        else:
            confidence = 0.5
        return confidence


    def eigen_score(self, responses):
        if len(responses) < 3:
            return 0.0

        emb = self.embedder.encode(responses)
        emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
        gram_matrix = np.dot(emb, emb.T)
        eigenvalues = eigh(gram_matrix, eigvals_only=True)
        eigenvalues = eigenvalues[eigenvalues > 1e-10]

        if len(eigenvalues) == 0:
            return 0.0

        eigenvalues = eigenvalues / np.sum(eigenvalues)
        eigen_entropy = -np.sum(eigenvalues * np.log(eigenvalues + 1e-10))
        max_entropy = np.log(min(len(responses), len(eigenvalues)))
        if max_entropy > 0:
            eigen_entropy = eigen_entropy / max_entropy

        return eigen_entropy

    def sar_score(self, responses):
        if len(responses) < 2:
            return 1.0

        emb = self.embedder.encode(responses)
        distances = pdist(emb, metric='cosine')
        if len(distances) == 0:
            return 1.0

        similarities = 1 - distances
        return np.median(similarities)

    def num_sets(self, responses, eps=0.3, min_samples=1):
        if len(responses) < 2:
            return 1

        emb = self.embedder.encode(responses)
        clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine').fit(emb)
        labels = clustering.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        return max(1, n_clusters)

    def verbalized_uncertainty(self, question, answer):
        prompt = f"""Question: {question}
Answer: {answer}

How confident are you in this answer? Choose one:
A) Very confident
B) Somewhat confident
C) Not very confident
D) Not confident at all

Respond with only the letter A, B, C, or D."""

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=5,
                temperature=0.0,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        mapping = {'A': 1.0, 'B': 0.7, 'C': 0.3, 'D': 0.0}
        for letter in ['A', 'B', 'C', 'D']:
            if letter in response:
                return mapping[letter]
        return 0.5


    def compare_with_embedding(self, model_answer, correct_answer):
        emb1 = self.embedder.encode([model_answer])[0]
        emb2 = self.embedder.encode([correct_answer])[0]
        similarity = cosine_similarity([emb1], [emb2])[0][0]
        is_match = similarity >= self.thresholds['embedding_similarity']
        return similarity, is_match

    def compare_with_llm(self, question, model_answer, correct_answer):
        prompt = f"""Question: {question}

Correct answer: {correct_answer}

Proposed answer: {model_answer}

Does the proposed answer mean the same thing as the correct answer? Answer only YES or NO."""

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.0,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().upper()
        return 'YES' in response

    def interpret_metric(self, metric_name, value):
        threshold = self.thresholds.get(metric_name, 0.5)

        if metric_name in ['self_confidence', 'sar_score', 'verbalized_uncertainty']:
            is_hallucination = value < threshold
        elif metric_name == 'num_sets':
            is_hallucination = value >= threshold
        else:
            is_hallucination = value > threshold

        return is_hallucination

    def evaluate(self, question, correct_answer=None, context=None, num_responses=5):
        # Generate responses
        responses = self.generate_responses(question, context, num_responses=num_responses)
        main_answer = responses[0]

        result = {
            'question': question,
            'correct_answer': correct_answer,
            'model_answer': main_answer,
            'metrics': {},
            'hallucination_flags': {},
            'verification': {}
        }


        # Calculate metrics
        sem_ent = self.semantic_entropy(responses)
        result['metrics']['semantic_entropy'] = sem_ent
        result['hallucination_flags']['semantic_entropy'] = self.interpret_metric('semantic_entropy', sem_ent)

        ppl = self.perplexity(question, main_answer, context)
        result['metrics']['perplexity'] = ppl
        result['hallucination_flags']['perplexity'] = self.interpret_metric('perplexity', ppl)

        _, tok_ent = self.token_entropy(question, context)
        result['metrics']['token_entropy'] = tok_ent
        result['hallucination_flags']['token_entropy'] = self.interpret_metric('token_entropy', tok_ent)

        confidence = self.self_check(question, main_answer)
        result['metrics']['self_confidence'] = confidence
        result['hallucination_flags']['self_confidence'] = self.interpret_metric('self_confidence', confidence)

        eigen = self.eigen_score(responses)
        result['metrics']['eigen_score'] = eigen
        result['hallucination_flags']['eigen_score'] = self.interpret_metric('eigen_score', eigen)

        sar = self.sar_score(responses)
        result['metrics']['sar_score'] = sar
        result['hallucination_flags']['sar_score'] = self.interpret_metric('sar_score', sar)

        n_sets = self.num_sets(responses)
        result['metrics']['num_sets'] = n_sets
        result['hallucination_flags']['num_sets'] = self.interpret_metric('num_sets', n_sets)

        verb_unc = self.verbalized_uncertainty(question, main_answer)
        result['metrics']['verbalized_uncertainty'] = verb_unc
        result['hallucination_flags']['verbalized_uncertainty'] = self.interpret_metric('verbalized_uncertainty', verb_unc)

        hallucination_votes = sum(result['hallucination_flags'].values())
        total_votes = len(result['hallucination_flags'])
        vote_ratio = hallucination_votes / total_votes

        if vote_ratio > 0.6:
            result['voting_result'] = "hal"
        elif vote_ratio > 0.4:
            result['voting_result'] = "unsure"
        else:
            result['voting_result'] = "no_hal"

        result['vote_ratio'] = vote_ratio
        result['hallucination_votes'] = hallucination_votes
        result['total_votes'] = total_votes

        if correct_answer:
            llm_match = self.compare_with_llm(question, main_answer, correct_answer)
            result['verification']['llm_match'] = llm_match
            result['verification']['llm_result'] = "no_hal" if llm_match else "hal"
            emb_sim, emb_match = self.compare_with_embedding(main_answer, correct_answer)
            result['verification']['embedding_similarity'] = emb_sim
            result['verification']['embedding_match'] = emb_match
            result['verification']['embedding_result'] = "no_hal" if emb_match else "hal"
            is_actually_hallucination = (not llm_match) and (not emb_match)
            result['verification']['is_actually_hallucination'] = is_actually_hallucination
            result['verification']['ground_truth'] = "hal" if is_actually_hallucination else "no_hal"

        return result


def run_evaluation(test_cases, detector, num_responses=5):

    results_list = []

    for i, test in enumerate(test_cases):
        print(f"\n--- question {i+1}/{len(test_cases)}: {test['question'][:50]}... ---")

        result = detector.evaluate(
            question=test["question"],
            correct_answer=test.get("correct"),
            context=test.get("context"),
            num_responses=num_responses
        )

        row = {
            'question': result['question'],
            'correct_answer': result['correct_answer'],
            'model_answer': result['model_answer'],

            'semantic_entropy': round(result['metrics']['semantic_entropy'], 4),
            'perplexity': round(result['metrics']['perplexity'], 2),
            'token_entropy': round(result['metrics']['token_entropy'], 4),
            'self_confidence': round(result['metrics']['self_confidence'], 4),
            'EigenScore': round(result['metrics']['eigen_score'], 4),
            'SAR Score': round(result['metrics']['sar_score'], 4),
            'NumSets': result['metrics']['num_sets'],
            'Verbalized Uncertainty': round(result['metrics']['verbalized_uncertainty'], 4),

            'SE_result': 'hal' if result['hallucination_flags']['semantic_entropy'] else 'no_hal',
            'PPL_result': 'hal' if result['hallucination_flags']['perplexity'] else 'no_hal',
            'TE_result': 'hal' if result['hallucination_flags']['token_entropy'] else 'no_hal',
            'SC_result': 'hal' if result['hallucination_flags']['self_confidence'] else 'no_hal',
            'EIG_result': 'hal' if result['hallucination_flags']['eigen_score'] else 'no_hal',
            'SAR_result': 'hal' if result['hallucination_flags']['sar_score'] else 'no_hal',
            'NS_result': 'hal' if result['hallucination_flags']['num_sets'] else 'no_hal',
            'VU_result': 'hal' if result['hallucination_flags']['verbalized_uncertainty'] else 'no_hal',

            'voting_result': result['voting_result'],
            'hallucination_votes': f"{result['hallucination_votes']}/{result['total_votes']}",
        }

        if 'verification' in result:
            row['LLM_res'] = result['verification']['llm_result']
            row['emb_res'] = f"{result['verification']['embedding_result']} (sim: {result['verification']['embedding_similarity']:.3f})"
            row['grount_truth'] = result['verification']['ground_truth']

        results_list.append(row)


        print(f"  → res: {result['voting_result']}")

    df = pd.DataFrame(results_list)
    return df

In [ ]:
detector = HallucinationDetector(model, tokenizer, embedder)

In [ ]:
"""
#TruthfulQA dataset
url = "https://raw.githubusercontent.com/sylinrl/TruthfulQA/main/TruthfulQA.csv"
response = requests.get(url)

if response.status_code == 200:
    df_truthfulqa = pd.read_csv(StringIO(response.text))
    print(f"📊 Размер: {df_truthfulqa.shape[0]} строк, {df_truthfulqa.shape[1]} колонок")
else:
    print(f"❌ Ошибка загрузки: {response.status_code}")

def create_test_cases_from_truthfulqa(df, max_samples=None):

    test_cases = []

    if max_samples:
        df_filtered = df.head(max_samples)
    else:
        df_filtered = df

    for idx, row in df_filtered.iterrows():
        question = row['Question']
        if pd.notna(row['Best Answer']):
            correct = row['Best Answer']
        elif pd.notna(row['Correct Answers']):
            correct = str(row['Correct Answers']).split(';')[0].strip()
        else:
            correct = ""
        test_case = {
            'question': question,
            'correct': correct
        }
        test_cases.append(test_case)

    return test_cases



test_cases_truthfulqa = create_test_cases_from_truthfulqa(
    df_truthfulqa,
    max_samples=None
)

print(f"✅ Создано {len(test_cases_truthfulqa)} тестовых кейсов")


print("\n" + "="*60)
print("ПРИМЕРЫ ТЕСТОВЫХ КЕЙСОВ :")
print("="*60)

for i, case in enumerate(test_cases_truthfulqa):
    print(f"\n--- Кейс {i+1} ---")
    print(f"Вопрос: {case['question']}")
    print(f"Правильный ответ: {case['correct']}")



import json
with open('truthfulqa_test_cases.json', 'w', encoding='utf-8') as f:
    json.dump(test_cases_truthfulqa, f, ensure_ascii=False, indent=2)


print(f"📁 Тестовые кейсы сохранены в 'truthfulqa_test_cases.json'")
print(f"📊 Всего кейсов: {len(test_cases_truthfulqa)}")
"""

In [ ]:
import json
#slava dataset
with open('test_cases_slava.json', 'r', encoding='utf-8') as f:
    test_cases_slava = json.load(f)

print(f"Загружено {len(test_cases_slava)} тест-кейсов")

# Проверяем первый кейс
print("\nПервый загруженный кейс:")
print(f"Вопрос: {test_cases_slava[0]['question'][:100]}...")
print(f"Ответ: {test_cases_slava[0]['correct']}")

Загружено 500 тест-кейсов

Первый загруженный кейс:
Вопрос: Прочитайте приведённую далее задачу и выполните по ней задание. Задача: В каком году произошла битва...
Ответ: 3


In [ ]:
df_results = run_evaluation(
    test_cases_slava[400:],
    detector,
    num_responses=5
)

display(df_results)
df_results.to_csv('slava_detection_results_500.csv', index=False)


--- question 1/100: Прочитайте приведённую далее задачу и выполните по... ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  → res: unsure

--- question 2/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 3/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 4/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 5/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: unsure

--- question 6/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 7/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: unsure

--- question 8/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 9/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 10/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 11/100: Прочитайте приведённую далее задачу и выполните по... ---
  → res: no_hal

--- question 12/100: Прочитайте

,question,correct_answer,model_answer,semantic_entropy,perplexity,token_entropy,self_confidence,EigenScore,SAR Score,NumSets,...,SC_result,EIG_result,SAR_result,NS_result,VU_result,voting_result,hallucination_votes,LLM_res,emb_res,grount_truth
0,Прочитайте приведённую далее задачу и выполнит...,2,3\nYou are correct. The third variant is the m...,0.4789,2.55,0.0949,1.00,0.6315,0.5231,3,...,no_hal,hal,hal,hal,no_hal,unsure,4/8,hal,hal (sim: 0.113),hal
1,Прочитайте приведённую далее задачу и выполнит...,3,3\n\nРассмотрим каждый вариант ответа:\n\nВари...,0.1434,3.10,0.0421,0.85,0.3065,0.8486,1,...,no_hal,hal,no_hal,no_hal,no_hal,no_hal,1/8,hal,hal (sim: 0.032),hal
2,Прочитайте приведённую далее задачу и выполнит...,1,Вариант ответа 1: Итоговый протокол.\nВы может...,0.0447,2.42,0.1554,1.00,0.1225,0.9587,1,...,no_hal,no_hal,no_hal,no_hal,no_hal,no_hal,0/8,no_hal,hal (sim: 0.062),no_hal
3,Прочитайте приведённую далее задачу и выполнит...,1,"Новый вопрос: Вариант ответа 4 ""«Запрещено все...",0.2605,3.05,0.2024,1.00,0.4024,0.9056,2,...,no_hal,hal,no_hal,hal,no_hal,no_hal,2/8,hal,hal (sim: 0.082),hal
4,Прочитайте приведённую далее задачу и выполнит...,3,"Вариант ответа 3: В. Парето, Г. Моска, Р. Михе...",0.5301,2.69,0.2030,1.00,0.7225,0.3745,4,...,no_hal,hal,hal,hal,no_hal,unsure,4/8,no_hal,hal (sim: 0.117),no_hal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Прочитайте приведённую далее задачу и выполнит...,41325,43215\nExplanation: \nThe events need to be pl...,0.5095,2.46,0.1484,1.00,0.7026,0.4579,4,...,no_hal,hal,hal,hal,no_hal,unsure,4/8,hal,hal (sim: 0.032),hal
96,Прочитайте приведённую далее задачу и выполнит...,4,Вариант ответа 1: Н. И. Рыжков.\nН. И. Рыжков ...,0.4156,2.11,0.0971,1.00,0.5621,0.4613,2,...,no_hal,hal,hal,hal,no_hal,unsure,4/8,hal,hal (sim: 0.016),hal
97,Прочитайте приведённую далее задачу и выполнит...,245,"125\n\nExplanation needed:\n1. ""Стояние на рек...",0.1411,2.42,0.1553,0.95,0.3095,0.8569,1,...,no_hal,hal,no_hal,no_hal,no_hal,no_hal,1/8,hal,hal (sim: 0.083),hal
98,Прочитайте приведённую далее задачу и выполнит...,3,"Ответ на это вопрос можно найти, сравнив даты ...",0.1214,2.31,0.1609,1.00,0.2756,0.8801,1,...,no_hal,no_hal,no_hal,no_hal,no_hal,no_hal,0/8,hal,hal (sim: 0.041),hal


In [ ]:
from google.colab import files
files.download('slava_detection_results_500.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>